In [1]:
import yfinance as yf
import pandas as pd
import numpy as np

# =========================
# CONFIGURAÇÕES
# =========================
ticker = "KEPL3.SA"

# Premissas macro (ajuste se quiser)
risk_free = 0.105      # 10.5%
market_premium = 0.055 # 5.5%
tax_rate = 0.34        # Brasil

# =========================
# 1. BAIXAR DADOS
# =========================
stock = yf.Ticker(ticker)

info = stock.info
balance = stock.balance_sheet
income = stock.financials

# =========================
# 2. VALOR DE MERCADO (E)
# =========================
market_cap = info.get("marketCap", np.nan)

# =========================
# 3. DÍVIDA (D)
# =========================
# Tentativa robusta (depende da disponibilidade)
try:
    short_debt = balance.loc["Short Long Term Debt"].iloc[0]
except:
    short_debt = 0

try:
    long_debt = balance.loc["Long Term Debt"].iloc[0]
except:
    long_debt = 0

try:
    cash = balance.loc["Cash"].iloc[0]
except:
    cash = 0

gross_debt = short_debt + long_debt
net_debt = gross_debt - cash

# =========================
# 4. CUSTO DA DÍVIDA (kd)
# =========================
try:
    interest_expense = income.loc["Interest Expense"].iloc[0]
    avg_debt = gross_debt if gross_debt != 0 else 1
    kd = abs(interest_expense) / avg_debt
except:
    kd = 0.12  # fallback

# =========================
# 5. BETA
# =========================
beta = info.get("beta", 1.0)

# =========================
# 6. CUSTO DE EQUITY (ke)
# =========================
ke = risk_free + beta * market_premium

# =========================
# 7. WACC
# =========================
D = max(net_debt, 0)
E = market_cap

wacc = (E / (D + E)) * ke + (D / (D + E)) * kd * (1 - tax_rate)

# =========================
# RESULTADOS
# =========================
print("===== WACC KEPL3 =====")
print(f"Market Cap (E): {E:,.0f}")
print(f"Dívida Bruta: {gross_debt:,.0f}")
print(f"Dívida Líquida: {net_debt:,.0f}")
print(f"Beta: {beta:.2f}")
print(f"Custo Equity (ke): {ke:.2%}")
print(f"Custo Dívida (kd): {kd:.2%}")
print(f"WACC: {wacc:.2%}")

===== WACC KEPL3 =====
Market Cap (E): 1,429,987,840
Dívida Bruta: 162,537,000
Dívida Líquida: 162,537,000
Beta: 0.18
Custo Equity (ke): 11.50%
Custo Dívida (kd): 30.02%
WACC: 12.35%
